#**Random Forest Model**

Import Library

In [98]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

from sklearn.metrics import mean_absolute_error, mean_squared_error

Load Data

In [82]:
df = pd.read_csv("online_food_sales_dataset.csv")

df.head()

,order_id,order_date,order_time,product_name,category,price,quantity,total_price,cost,profit,customer_id,customer_type,city,order_channel,payment_method,delivery_type,rating
0,ORD00001,2023-09-01,10:07:00,Chicken Wings,Appetizer,35000,1,35000,18000,17000,CUST0394,Returning,Semarang,Mobile App,Cash on Delivery,Standard,2
1,ORD00002,2023-07-04,13:22:00,Pizza Margherita,Main Course,85000,3,255000,135000,120000,CUST0053,Returning,Semarang,Website,E-Wallet,Standard,4
2,ORD00003,2023-10-25,16:02:00,Ice Cream Sundae,Dessert,25000,1,25000,10000,15000,CUST1359,Returning,Semarang,Mobile App,E-Wallet,Standard,3
3,ORD00004,2023-10-27,09:32:00,Iced Latte,Beverage,35000,4,140000,60000,80000,CUST0577,New,Jakarta,Mobile App,Cash on Delivery,Express,1
4,ORD00005,2023-04-02,19:17:00,Chocolate Brownie,Dessert,35000,1,35000,15000,20000,CUST0067,Returning,Semarang,Mobile App,Credit Card,Scheduled,4


Data Understanding

In [83]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   order_id        3000 non-null   object
 1   order_date      3000 non-null   object
 2   order_time      3000 non-null   object
 3   product_name    3000 non-null   object
 4   category        3000 non-null   object
 5   price           3000 non-null   int64 
 6   quantity        3000 non-null   int64 
 7   total_price     3000 non-null   int64 
 8   cost            3000 non-null   int64 
 9   profit          3000 non-null   int64 
 10  customer_id     3000 non-null   object
 11  customer_type   3000 non-null   object
 12  city            3000 non-null   object
 13  order_channel   3000 non-null   object
 14  payment_method  3000 non-null   object
 15  delivery_type   3000 non-null   object
 16  rating          3000 non-null   int64 
dtypes: int64(6), object(11)
memory usage: 398.6+ KB


,0
order_id,0
order_date,0
order_time,0
product_name,0
category,0
price,0
quantity,0
total_price,0
cost,0
profit,0


Aggreagate Data

In [84]:
#Agreggate Data
df['order_date'] = pd.to_datetime(df['order_date'])

df_daily = df.groupby(['order_date', 'product_name']).agg({
    'quantity': 'sum',
    'price': 'mean',
    'order_channel': 'count'
}).reset_index()

#Rename
df_daily.rename(columns={
    'quantity': 'daily_demand',
    'order_channel': 'transaction_count'
}, inplace=True)

Feature Engineering

In [85]:
df_daily['day'] = df_daily['order_date'].dt.day
df_daily['month'] = df_daily['order_date'].dt.month
df_daily['day_of_week'] = df_daily['order_date'].dt.dayofweek
df_daily['is_weekend'] = df_daily['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

Select Feature & Target

In [86]:
X = df_daily[[
    'product_name',
    'price',
    'transaction_count',
    'day',
    'month',
    'day_of_week',
    'is_weekend'
]]

y = df_daily['daily_demand']

Encoding (One Hot Encoding)

In [87]:
categorical = ['product_name']
numerical = ['price', 'transaction_count', 'day', 'month', 'day_of_week', 'is_weekend']

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical)
], remainder='passthrough')

Modelling

In [88]:
model = Pipeline([
    ('prep', preprocessor),
    ('rf', RandomForestRegressor(n_estimators=100, random_state=42))
])

Data Split and Training

In [89]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('prep',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['product_name'])])),
                ('rf', RandomForestRegressor(random_state=42))])

Evaluation

In [96]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("MAE", mae)
print("RMSE", rmse)

MAE 1.460135746606335
RMSE 1.7620307853972879


In [99]:
mape = mean_absolute_percentage_error(y_test, y_pred)
print(mape)

0.5813789195344625


In [100]:
relative_error = mae / mean_demand
print(relative_error)

0.35377314664180276


**#Random Forest Model Interpretation to Daily Demand Forecasting**

The model achieved an average error of approximately 1–2 units per day per product, corresponding to a 35% relative error. The higher MAPE value (58%) is influenced by the small demand scale, where minor deviations result in large percentage errors.

#**LINEAR REGRESSION MODEL**

Modelling

In [101]:
baseline_model = Pipeline([
    ('prep', preprocessor),
    ('lr', LinearRegression())
])

Train Data

In [102]:
baseline_model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('prep',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['product_name'])])),
                ('lr', LinearRegression())])

Predict & Evaluation

In [103]:
y_pred_lr = baseline_model.predict(X_test)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))

print("Linear Regression MAE:", mae_lr)
print("Linear Regression RMSE:", rmse_lr)

Linear Regression MAE: 1.3863272124419224
Linear Regression RMSE: 1.6381070974395606


In [105]:
mape_lr = mean_absolute_percentage_error(y_test, y_pred_lr)
print("MAPE:", mape_lr)

MAPE: 0.5653364484807761


In [106]:
mae_lr = mean_absolute_error(y_test, y_pred_lr)

relative_error_lr = mae_lr / mean_demand
print("Relative Error:", relative_error_lr)

Relative Error: 0.33589023579529287


**#Linear Regression Model Interpretation to Daily Demand Forecasting**

The Linear Regression model predicts daily product demand with an average error of approximately 1–2 units per day per product. A relative error of 34% indicates acceptable performance, while the higher MAPE value (57%) is influenced by the small demand scale, where minor deviations lead to large percentage errors.

#**Final Insight**

A comparison between Linear Regression and Random Forest shows that Linear Regression consistently outperformed the more complex model across all evaluation metrics. This indicates that the relationship between features and demand is largely linear, and additional model complexity does not provide significant benefits. Despite relatively high MAPE values due to low demand scale, both models demonstrate stable prediction performance, with Linear Regression being the most efficient and interpretable choice.